In [1]:
# data 
from __future__ import annotations

from enum import Enum
from typing import Any, Callable, Optional, Protocol, Sequence, Union, runtime_checkable

import numpy as np
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, field_validator


# --------------------------------------------------------------------------- #
# Model & loader protocols                                                     #
# --------------------------------------------------------------------------- #
@runtime_checkable
class Estimator(Protocol):
    """Anything scikit-learn-shaped."""

    def fit(self, X: Any, y: Any) -> Any: ...
    def predict(self, X: Any) -> Any: ...


LoaderLike = Union[pd.DataFrame, Callable[[], pd.DataFrame], Any]


def _as_frame(loader: LoaderLike) -> pd.DataFrame:
    """Coerce a 'pydantic dataloader' (or anything reasonable) into a DataFrame."""
    if isinstance(loader, pd.DataFrame):
        return loader.copy()
    if callable(loader):
        return _as_frame(loader())
    for attr in ("load", "to_frame", "dataframe", "df", "data"):
        if hasattr(loader, attr):
            obj = getattr(loader, attr)
            obj = obj() if callable(obj) else obj
            if isinstance(obj, pd.DataFrame):
                return obj.copy()
    # pydantic v2 BaseModel list -> DataFrame
    if hasattr(loader, "model_dump"):
        return pd.json_normalize(loader.model_dump())
    if isinstance(loader, Sequence) and len(loader) and hasattr(loader[0], "model_dump"):
        return pd.DataFrame([r.model_dump() for r in loader])
    raise TypeError(
        f"Could not turn {type(loader)!r} into a DataFrame. Pass a DataFrame, a "
        "callable returning one, or an object with .load()/.to_frame()/.dataframe."
    )


class Task(str, Enum):
    CLASSIFICATION = "classification"
    REGRESSION = "regression"


# --------------------------------------------------------------------------- #
# Config (pydantic) — the 'no-code' knobs live here                            #
# --------------------------------------------------------------------------- #
class EvalConfig(BaseModel):
    """All tunables in one validated place; every analyzer accepts one."""

    model_config = ConfigDict(extra="forbid")

    task: Task = Task.CLASSIFICATION
    random_state: int = 42
    n_repeats: int = 12            # Monte-Carlo repeats (edf, stability, rademacher)
    n_bootstrap: int = 20          # bootstrap resamples for prediction variance
    max_rows_for_refit: int = 4000  # subsample cap to keep refit-heavy stats cheap
    cv_folds: int = 4
    # weak-point discovery
    max_slice_depth: int = 3       # depth of the interpretable "error tree"
    min_slice_frac: float = 0.03   # ignore slices smaller than this share of test
    # temporal / drift
    time_col: Optional[str] = None
    n_windows: int = 8

    @field_validator("min_slice_frac")
    @classmethod
    def _frac(cls, v):
        if not 0 < v < 1:
            raise ValueError("min_slice_frac must be in (0, 1)")
        return v


# --------------------------------------------------------------------------- #
# Dataset — the single object the toolkit consumes                             #
# --------------------------------------------------------------------------- #
class Dataset(BaseModel):
    """Immutable-ish container. Holds numeric feature matrices + targets."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    X_train: pd.DataFrame
    y_train: pd.Series
    X_test: pd.DataFrame
    y_test: pd.Series
    feature_names: list[str]
    task: Task = Task.CLASSIFICATION
    time_train: Optional[pd.Series] = None
    time_test: Optional[pd.Series] = None

    # ---- constructors ---------------------------------------------------- #
    @classmethod
    def from_frames(
        cls,
        train: pd.DataFrame,
        test: pd.DataFrame,
        target: str,
        task: Union[Task, str] = Task.CLASSIFICATION,
        time_col: Optional[str] = None,
        features: Optional[Sequence[str]] = None,
    ) -> "Dataset":
        task = Task(task)
        drop = {target} | ({time_col} if time_col else set())
        feats = list(features) if features else [c for c in train.columns if c not in drop]
        return cls(
            X_train=train[feats].reset_index(drop=True),
            y_train=train[target].reset_index(drop=True),
            X_test=test[feats].reset_index(drop=True),
            y_test=test[target].reset_index(drop=True),
            feature_names=feats,
            task=task,
            time_train=(train[time_col].reset_index(drop=True) if time_col else None),
            time_test=(test[time_col].reset_index(drop=True) if time_col else None),
        )

    @classmethod
    def from_loaders(
        cls,
        train_loader: LoaderLike,
        test_loader: LoaderLike,
        target: str,
        task: Union[Task, str] = Task.CLASSIFICATION,
        time_col: Optional[str] = None,
        features: Optional[Sequence[str]] = None,
    ) -> "Dataset":
        """Entry point for a pydantic dataloader (or any DataFrame-yielding thing)."""
        return cls.from_frames(
            _as_frame(train_loader), _as_frame(test_loader),
            target=target, task=task, time_col=time_col, features=features,
        )

    # ---- convenience ----------------------------------------------------- #
    @property
    def n_train(self) -> int:
        return len(self.X_train)

    @property
    def n_test(self) -> int:
        return len(self.X_test)

    def numpy(self):
        return (
            self.X_train.to_numpy(dtype=float),
            self.y_train.to_numpy(),
            self.X_test.to_numpy(dtype=float),
            self.y_test.to_numpy(),
        )

    def subsample_train(self, n: int, rng: np.random.Generator):
        """Return (Xs, ys) subsample of train for cheap refit-heavy stats."""
        if self.n_train <= n:
            return self.X_train.to_numpy(float), self.y_train.to_numpy()
        idx = rng.choice(self.n_train, size=n, replace=False)
        return self.X_train.to_numpy(float)[idx], self.y_train.to_numpy()[idx]


In [2]:
"""Metric primitives shared across analyzers (kept tiny and dependency-light)."""
from __future__ import annotations

import numpy as np
from sklearn.metrics import log_loss, roc_auc_score


def per_sample_loss(y_true, model, X, task: Task) -> np.ndarray:
    """Per-row loss: log-loss (clf) or squared error (reg). Lower = better."""
    if task == Task.CLASSIFICATION:
        proba = _proba(model, X)
        eps = 1e-12
        yt = np.asarray(y_true).astype(int)
        p = np.clip(proba[np.arange(len(yt)), yt], eps, 1 - eps)
        return -np.log(p)
    pred = np.asarray(model.predict(X), dtype=float)
    return (np.asarray(y_true, dtype=float) - pred) ** 2


def score(y_true, model, X, task: Task) -> float:
    """A single 'higher is better' headline score: AUC (clf) or R2 (reg)."""
    if task == Task.CLASSIFICATION:
        proba = _proba(model, X)[:, 1] if _proba(model, X).shape[1] == 2 else None
        try:
            if proba is not None:
                return float(roc_auc_score(y_true, proba))
            return float(roc_auc_score(y_true, _proba(model, X), multi_class="ovr"))
        except Exception:
            return float((np.asarray(model.predict(X)) == np.asarray(y_true)).mean())
    pred = np.asarray(model.predict(X), dtype=float)
    yt = np.asarray(y_true, dtype=float)
    ss_res = np.sum((yt - pred) ** 2)
    ss_tot = np.sum((yt - yt.mean()) ** 2) + 1e-12
    return float(1 - ss_res / ss_tot)


def _proba(model, X) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return np.asarray(model.predict_proba(X), dtype=float)
    if hasattr(model, "decision_function"):
        d = np.asarray(model.decision_function(X), dtype=float)
        if d.ndim == 1:
            p = 1 / (1 + np.exp(-d))
            return np.column_stack([1 - p, p])
        e = np.exp(d - d.max(axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)
    pred = np.asarray(model.predict(X)).astype(int)
    n_cls = int(pred.max()) + 1
    oh = np.zeros((len(pred), max(n_cls, 2)))
    oh[np.arange(len(pred)), pred] = 1.0
    return oh


def headline_name(task: Task) -> str:
    return "AUC" if task == Task.CLASSIFICATION else "R2"


In [3]:
# from diagnostics suite
from __future__ import annotations

from typing import Literal, Optional

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import ks_2samp, wasserstein_distance

def _scalar_output(model, X, task: Task) -> np.ndarray:
    if task == Task.CLASSIFICATION:
        p = _proba(model, X)
        return p[:, 1] if p.shape[1] == 2 else p.max(axis=1)
    return np.asarray(model.predict(X), dtype=float)


# --------------------------------------------------------------------------- #
# 1) INPUT SENSITIVITY  (local Lipschitz)  — a refit-free complexity signal    #
#    Modeva: overfit models have larger ||grad f|| / higher local Lipschitz.   #
#    Cheaper substitute/complement for our bootstrap prediction_variance.      #
# --------------------------------------------------------------------------- #
def input_sensitivity(model, X: np.ndarray, task: Task, *, n_dirs: int = 8,
                      h: float = 0.05, random_state: int = 0) -> dict:
    """Estimate mean local Lipschitz constant E[ |f(x+δ)-f(x)| / ||δ|| ] using
    random directions (SPSA-style). No refit: n_dirs forward passes only.

    Returns {'lipschitz_mean', 'lipschitz_p95'} on the model's scalar output.
    """
    rng = np.random.default_rng(random_state)
    X = np.asarray(X, dtype=float)
    sd = X.std(axis=0) + 1e-9
    f0 = _scalar_output(model, X, task)
    out_scale = float(np.std(f0)) + 1e-9
    ratios = np.zeros((n_dirs, len(X)))
    abs_moves = np.zeros(n_dirs)
    for k in range(n_dirs):
        delta = rng.normal(0, 1, size=X.shape) * (h * sd)
        fh = _scalar_output(model, X + delta, task)
        norm = np.linalg.norm(delta, axis=1) + 1e-12
        ratios[k] = np.abs(fh - f0) / norm
        abs_moves[k] = np.mean(np.abs(fh - f0))
    per_sample = ratios.mean(axis=0)
    # unit-free: how much of the output's own spread an h-perturbation moves
    relative_jaggedness = float(abs_moves.mean() / out_scale)
    return dict(lipschitz_mean=float(per_sample.mean()),
                lipschitz_p95=float(np.percentile(per_sample, 95)),
                relative_jaggedness=relative_jaggedness,
                per_sample=per_sample)


# --------------------------------------------------------------------------- #
# 2) DRIFT DRIVERS  — which FEATURES shift between two windows                  #
#    Modeva 'resilience': rank features by PSI / KS / Wasserstein.             #
#    For our temporal_curve: explain WHAT drifts once drift is detected.       #
# --------------------------------------------------------------------------- #
def _psi(ref: np.ndarray, cur: np.ndarray, bins: int = 10) -> float:
    edges = np.quantile(ref, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    eps = 1e-6
    e = np.histogram(ref, edges)[0] / (len(ref) + eps) + eps
    a = np.histogram(cur, edges)[0] / (len(cur) + eps) + eps
    return float(np.sum((a - e) * np.log(a / e)))


def drift_drivers(X_ref: pd.DataFrame, X_cur: pd.DataFrame,
                  feature_names: list[str], top: Optional[int] = None) -> pd.DataFrame:
    """Rank features by distribution shift between a reference and current window.
    Cheap: histograms + sort, no model calls."""
    rows = []
    for f in feature_names:
        r = X_ref[f].to_numpy(float)
        c = X_cur[f].to_numpy(float)
        rows.append(dict(
            feature=f,
            psi=round(_psi(r, c), 3),
            ks=round(float(ks_2samp(r, c).statistic), 3),
            wasserstein=round(float(wasserstein_distance(r, c)), 3),
            mean_shift=round(float(c.mean() - r.mean()), 3),
        ))
    df = pd.DataFrame(rows).sort_values("psi", ascending=False).reset_index(drop=True)
    df["severity"] = pd.cut(df["psi"], [-1, 0.1, 0.25, np.inf],
                            labels=["low", "moderate", "high"])
    return df.head(top) if top else df


# --------------------------------------------------------------------------- #
# 3) ADAPTIVE WEAK THRESHOLD  — Modeva: flag regions where gap > mu + beta*std #
#    Replaces our fixed loss_ratio>1.25 gate with a data-driven one. Free.     #
# --------------------------------------------------------------------------- #
def adaptive_weak_threshold(values: np.ndarray, beta: float = 1.5) -> float:
    values = np.asarray(values, dtype=float)
    return float(np.mean(values) + beta * np.std(values))


# --------------------------------------------------------------------------- #
# 4) RESILIENCE STRESS CURVE  — Modeva 'resilience' worst-case drift scenarios  #
#    Refit-free stress test that needs NO time column (complements temporal).  #
# --------------------------------------------------------------------------- #
Scenario = Literal["worst-sample", "worst-cluster", "edge", "hard-sample"]


def resilience_stress_curve(model, ds: Dataset, *, scenario: Scenario = "worst-sample",
                            levels=(0.0, 0.1, 0.2, 0.3, 0.4, 0.5),
                            n_clusters: int = 5, random_state: int = 0):
    """Progressively oversample a 'hard' subpopulation of the TEST set and track
    the metric — a degradation curve. No refit; only predictions + one cheap
    selection step (KMeans or covariance inverse).

    Returns (DataFrame, plotly.Figure, verdict_str).
    """
    Xte = ds.X_test.to_numpy(float)
    yte = ds.y_test.to_numpy()
    n = len(yte)
    rng = np.random.default_rng(random_state)

    # rank samples by "hardness" for the chosen scenario (higher = harder)
    if scenario == "worst-sample":
        hard = per_sample_loss(yte, model, Xte, ds.task)
    elif scenario == "hard-sample":
        if ds.task == Task.CLASSIFICATION:
            p = _proba(model, Xte)
            hard = 1.0 - np.sort(p, axis=1)[:, -1]      # low confidence = hard
        else:
            hard = per_sample_loss(yte, model, Xte, ds.task)
    elif scenario == "edge":
        mu = Xte.mean(axis=0)
        cov = np.cov(Xte, rowvar=False) + 1e-6 * np.eye(Xte.shape[1])
        inv = np.linalg.pinv(cov)
        d = Xte - mu
        hard = np.einsum("ij,jk,ik->i", d, inv, d)      # Mahalanobis^2
    elif scenario == "worst-cluster":
        from sklearn.cluster import KMeans
        from sklearn.preprocessing import StandardScaler
        km = KMeans(n_clusters=n_clusters, n_init=5, random_state=random_state)
        lab = km.fit_predict(StandardScaler().fit_transform(Xte))
        loss = per_sample_loss(yte, model, Xte, ds.task)
        worst = max(range(n_clusters), key=lambda c: loss[lab == c].mean()
                    if (lab == c).any() else -np.inf)
        hard = (lab == worst).astype(float) + rng.uniform(0, 1e-6, n)
    else:
        raise ValueError(scenario)

    hard_order = np.argsort(-hard)
    base_metric = score(yte, model, Xte, ds.task)
    rows = []
    for alpha in levels:
        n_hard = int(alpha * n)
        idx = np.concatenate([hard_order[:n_hard],
                              rng.choice(n, size=n - n_hard, replace=True)])
        rows.append(dict(drift_level=alpha,
                         metric=round(score(yte[idx], model, Xte[idx], ds.task), 4)))
    curve = pd.DataFrame(rows)

    name = headline_name(ds.task)
    worst_metric = curve["metric"].iloc[-1]
    degradation = base_metric - worst_metric
    fig = go.Figure(go.Scatter(x=curve["drift_level"], y=curve["metric"],
                               mode="lines+markers", line=dict(color=viz.ACCENT)))
    fig.add_hline(y=base_metric, line=dict(color=MUTED, dash="dot"),
                  annotation_text="baseline")
    fig.update_xaxes(title=f"fraction drifted toward '{scenario}'")
    fig.update_yaxes(title=name)
    fig = _base(fig, f"Resilience stress curve — {scenario}", height=420)

    rel = degradation / (abs(base_metric) + 1e-9)
    verdict = (f"Under '{scenario}' drift, {name} falls {degradation:.3f} "
               f"({rel:.0%}) from baseline {base_metric:.3f} to {worst_metric:.3f} "
               f"at 50% contamination. "
               + ("LOW resilience — the model degrades sharply toward this "
                  "subpopulation." if rel > 0.15 else
                  "Reasonable resilience to this scenario."))
    return curve, fig, verdict


In [4]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

In [5]:
# 
"""Shared plotly helpers + a consistent theme. Everything the toolkit draws
comes through here, so the look is uniform and easy to restyle."""
from __future__ import annotations

from typing import Optional, Sequence

import pandas as pd
import plotly.graph_objects as go

# One place to control the palette.
INK = "#1f2733"
MUTED = "#8a94a6"
GRID = "#e9edf3"
ACCENT = "#2f6df6"
GOOD = "#2fa36b"
WARN = "#e8a13a"
BAD = "#d1495b"
SEQ = ["#2f6df6", "#8a4fff", "#2fa36b", "#e8a13a", "#d1495b", "#00b3b3"]


def _base(fig: go.Figure, title: str, height: int = 420) -> go.Figure:
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=INK)),
        template="plotly_white",
        font=dict(family="Inter, Segoe UI, sans-serif", color=INK, size=12),
        height=height,
        margin=dict(l=60, r=30, t=60, b=50),
        legend=dict(bgcolor="rgba(255,255,255,0.6)", borderwidth=0),
    )
    fig.update_xaxes(gridcolor=GRID, zeroline=False)
    fig.update_yaxes(gridcolor=GRID, zeroline=False)
    return fig


def table(df: pd.DataFrame, title: str, color_col: Optional[str] = None,
          height: Optional[int] = None) -> go.Figure:
    """Render a DataFrame as an interactive plotly table."""
    fill = "white"
    if color_col and color_col in df.columns:
        vals = df[color_col]
        fill = [[c] * len(df) if col != color_col else list(vals) for col in df.columns]
    header_vals = [f"<b>{c}</b>" for c in df.columns]
    cells = [df[c].tolist() for c in df.columns]
    fig = go.Figure(go.Table(
        header=dict(values=header_vals, fill_color="#f2f5fa",
                    align="left", font=dict(color=INK, size=12), height=30),
        cells=dict(values=cells, align="left",
                   fill_color=[fill] if isinstance(fill, str) else fill,
                   font=dict(color=INK, size=11), height=26),
    ))
    fig.update_layout(title=dict(text=title, font=dict(size=16, color=INK)),
                      margin=dict(l=10, r=10, t=50, b=10),
                      height=height or (60 + 28 * (len(df) + 1)))
    return fig


def bars(labels: Sequence[str], values: Sequence[float], title: str,
         colors: Optional[Sequence[str]] = None, ytitle: str = "") -> go.Figure:
    fig = go.Figure(go.Bar(
        x=list(labels), y=list(values),
        marker_color=colors or ACCENT,
        text=[f"{v:.3g}" for v in values], textposition="outside",
    ))
    fig.update_yaxes(title=ytitle)
    return _base(fig, title, height=380)


def radar(labels: Sequence[str], values: Sequence[float], title: str) -> go.Figure:
    labels = list(labels) + [labels[0]]
    values = list(values) + [values[0]]
    fig = go.Figure(go.Scatterpolar(r=values, theta=labels, fill="toself",
                                    line_color=ACCENT))
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=INK)),
        polar=dict(radialaxis=dict(range=[0, 1], gridcolor=GRID)),
        template="plotly_white", height=440,
        margin=dict(l=60, r=60, t=60, b=40),
    )
    return fig


In [6]:
CAUSE_COLOR = {
    "Local overfitting": BAD,
    "Data sparsity": WARN,
    "Distribution shift": "#8a4fff",
    "Irreducible noise": MUTED,
    "Hard / underfit": "#c86b00",
    "Elevated error": "#5b6b82",
    "(healthy)": GOOD,
}


In [ ]:
# import pandas as pd

In [40]:

@dataclass
class WeakpointReport:
    slices: pd.DataFrame
    #figures_: dict = field(default_factory=dict)
    verdict: str = ""
    #details: dict = field(default_factory=dict)

    def figures(self) -> dict[str, go.Figure]:
        return self.figures_

    def show(self):
        for f in self.figures_.values():
            f.show()


class WeakpointAnalyzer:
    def __init__(self, config: Optional[EvalConfig] = None):
        self.cfg = config or EvalConfig()

    # ---- public ---------------------------------------------------------- #
    def analyze(self, model, ds: Dataset) -> WeakpointReport:
        feats = ds.feature_names
        Xtr, Xte = ds.X_train.to_numpy(float), ds.X_test.to_numpy(float)
        ytr, yte = ds.y_train.to_numpy(), ds.y_test.to_numpy()

        loss_te = per_sample_loss(yte, model, Xte, ds.task)
        loss_tr = per_sample_loss(ytr, model, Xtr, ds.task)
        g_loss_te = float(loss_te.mean())
        g_loss_tr = float(loss_tr.mean())
        g_score = score(yte, model, Xte, ds.task)

        # ---- discover slices via an error tree --------------------------- #
        etree = DecisionTreeRegressor(
            max_depth=self.cfg.max_slice_depth,
            min_samples_leaf=max(20, int(self.cfg.min_slice_frac * len(yte))),
            random_state=self.cfg.random_state,
        ).fit(Xte, loss_te)
        leaf_te = etree.apply(Xte)
        leaf_tr = etree.apply(Xtr)
        rules = _leaf_rules(etree, feats)

        # # neighbor structure for label-noise proxy AND density axis (fit once)
        # scaler = StandardScaler().fit(Xte)
        # nn = NearestNeighbors(n_neighbors=min(11, len(yte))).fit(scaler.transform(Xte))
        # dist, nbr_idx = nn.kneighbors(scaler.transform(Xte))
        # noise_all = self._neighbor_disagreement(yte, nbr_idx, ds.task)
        # # AMIF-style density axis (cheap): mean distance to neighbours; larger = sparser
        # density_all = dist[:, 1:].mean(axis=1)
        # g_density = float(np.median(density_all) + 1e-12)

        rows = []
        for lid in np.unique(leaf_te):
            m_te = leaf_te == lid
            if m_te.sum() < max(10, int(0.5 * self.cfg.min_slice_frac * len(yte))):
                continue
            m_tr = leaf_tr == lid
            # row = self._diagnose_slice(
            #     ds, model, m_te, m_tr, loss_te, loss_tr, yte, Xte,
            #     g_loss_te, g_loss_tr, g_score, noise_all, density_all, g_density,
            #     rules.get(lid, "all"))
            # rows.append(row)
            row = self._diagnose_slice(
                ds, model, m_te, m_tr, loss_te, loss_tr, yte, Xte,
                g_loss_te, g_loss_tr, g_score,
                rules.get(lid, "all"))
            rows.append(row)

        slices = pd.DataFrame(rows).sort_values("loss_ratio", ascending=False)
        slices = slices.reset_index(drop=True)

        # ADAPTIVE threshold (Modeva: mu + beta*sigma over regions), kept in a
        # sensible band: never flag a region <10% worse, always flag one >=50% worse.
        if len(slices) > 2:
            raw = adaptive_weak_threshold(slices["loss_ratio"].to_numpy(), beta=1.0)
            thr = float(min(1.5, max(1.1, raw)))
        else:
            thr = 1.25
        slices["likely_cause"] = [
            self._attribute(r, g_loss_te, g_loss_tr, thr) for _, r in slices.iterrows()]

        # error drivers = feature importances of the error tree (free — already fit)
        drivers = pd.DataFrame({"feature": feats,
                                "importance": np.round(etree.feature_importances_, 3)}) \
            .sort_values("importance", ascending=False).reset_index(drop=True)

        # figs = {
        #     "table": self._table_fig(slices),
        #     "map": self._embedding_fig(Xte, loss_te, leaf_te, slices, feats, Xte),
        #     "landscape": self._landscape_fig(density_all, noise_all, loss_te,
        #                                      g_density),
        #     "drivers": self._drivers_fig(drivers),
        #     "profile": self._feature_profile_fig(ds, loss_te),
        # }
        verdict = self._verdict(slices, g_score, ds.task, thr)
        return WeakpointReport(slices=slices,verdict=verdict), drivers

    # ---- per-slice diagnostics ------------------------------------------ #
    def _diagnose_slice(self, ds, model, m_te, m_tr, loss_te, loss_tr, yte, Xte,
                        g_loss_te, g_loss_tr, g_score, rule) -> dict:
        n_te = int(m_te.sum())
        n_tr = int(m_tr.sum())
        test_frac = n_te / len(loss_te)
        train_frac = n_tr / len(loss_tr)

        local_loss_te = float(loss_te[m_te].mean())
        local_loss_tr = float(loss_tr[m_tr].mean()) if n_tr > 0 else np.nan
        loss_ratio = local_loss_te / (g_loss_te + 1e-12)
        overfit_gap = (local_loss_te - local_loss_tr) if n_tr > 0 else np.nan
        coverage = (train_frac / (test_frac + 1e-12))
        #noise = float(noise_all[m_te].mean())
        #density_ratio = float(density_all[m_te].mean() / g_density)   # >1 = sparser
        loss_std = float(loss_te[m_te].std())                         # Modeva Uncertainty(R)
        pred_var = float(np.var(_scalar_output(model, Xte[m_te], ds.task))) \
            if n_te > 1 else np.nan                                   # Modeva LocalComplexity(R)
        can_score = n_te > 5 and (ds.task == Task.REGRESSION or
                                   len(np.unique(yte[m_te])) > 1)
        local_score = score(yte[m_te], model, Xte[m_te], ds.task) if can_score else np.nan
        margin = self._uncertainty(model, Xte[m_te], ds.task)

        return dict(
            rule=rule, n_test=n_te, test_share=round(test_frac, 3),
            train_share=round(train_frac, 3),
            local_score=round(float(local_score), 3) if local_score == local_score else np.nan,
            global_score=round(g_score, 3),
            local_test_loss=round(local_loss_te, 3),
            local_train_loss=round(local_loss_tr, 3) if n_tr else np.nan,
            loss_ratio=round(loss_ratio, 2),
            overfit_gap=round(overfit_gap, 3) if n_tr else np.nan,
            coverage=round(coverage, 2),
            local_pred_var=round(pred_var, 4),
            loss_std=round(loss_std, 3), uncertainty=round(margin, 3),
        )

    def _attribute(self, r, g_te, g_tr, thr) -> str:
        """Transparent rule set over a slice row. `thr` is the adaptive
        (mu+beta*sigma) loss-ratio threshold computed across regions."""
        if r["loss_ratio"] < thr:
            return "(healthy)"
        l_te, l_tr = r["local_test_loss"], r["local_train_loss"]
        overfit_gap = r["overfit_gap"]
        global_gap = g_te - g_tr
        # 1) local overfitting: test loss >> train loss HERE, beyond the global gap
        if overfit_gap == overfit_gap and overfit_gap > max(2 * global_gap, 0.15) \
                and (l_tr < 0.8 * g_te):
            return "Local overfitting"
        # 2) sparsity: few train points OR AMIF density axis says region is sparse
        # if r["train_share"] < 0.4 * r["test_share"] or r["coverage"] < 0.5 \
        #         or r["density_ratio"] > 1.5:
        if r["train_share"] < 0.4 * r["test_share"] or r["coverage"] < 0.5:
            return "Data sparsity"
        # 3) covariate shift: region much bigger at test time than train time
        if r["test_share"] > 1.8 * r["train_share"]:
            return "Distribution shift"
        # 4) irreducible: neighbours disagree a lot => high Bayes error
        # if r["label_noise"] > 0.35:
        #     return "Irreducible noise"
        # 5) hard/underfit: train ALSO bad here
        if l_tr == l_tr and l_tr > 1.3 * g_tr:
            return "Hard / underfit"
        return "Elevated error"

    # @staticmethod
    # def _neighbor_disagreement(y, nbr_idx, task: Task) -> np.ndarray:
    #     y = np.asarray(y)
    #     neigh = y[nbr_idx[:, 1:]]  # drop self
    #     if task == Task.CLASSIFICATION:
    #         return (neigh != y[:, None]).mean(axis=1)
    #     # regression: coefficient of variation of neighbor targets, clipped
    #     mu = neigh.mean(axis=1)
    #     sd = neigh.std(axis=1)
    #     return np.clip(sd / (np.abs(mu) + np.std(y) + 1e-9), 0, 1)

    @staticmethod
    def _uncertainty(model, X, task: Task) -> float:
        if len(X) == 0:
            return np.nan
        if task == Task.CLASSIFICATION:
            p = _proba(model, X)
            top = np.sort(p, axis=1)[:, -1]
            return float(1 - np.mean(top))  # closeness to a coin flip
        return float(np.std(np.asarray(model.predict(X), float)))

    # ---- figures --------------------------------------------------------- #
    def _table_fig(self, slices: pd.DataFrame) -> go.Figure:
        show = slices[["rule", "n_test", "test_share", "train_share", "local_score",
                       "loss_ratio", "overfit_gap", "coverage", "likely_cause"]].copy()
        colors = [CAUSE_COLOR.get(c, "white") for c in show["likely_cause"]]
        # tint only the cause column
        fill = []
        for col in show.columns:
            if col == "likely_cause":
                fill.append([_tint(c) for c in colors])
            else:
                fill.append(["white"] * len(show))
        fig = go.Figure(go.Table(
            header=dict(values=[f"<b>{c}</b>" for c in show.columns],
                        fill_color="#f2f5fa", align="left",
                        font=dict(color=INK, size=11), height=30),
            cells=dict(values=[show[c].tolist() for c in show.columns],
                       fill_color=fill, align="left",
                       font=dict(color=INK, size=10.5), height=26),
        ))
        fig.update_layout(title=dict(text="Weak regions ranked by loss ratio (vs global)",
                                     font=dict(size=16, color=INK)),
                          margin=dict(l=10, r=10, t=50, b=10),
                          height=70 + 28 * (len(show) + 1))
        return fig


    def _feature_profile_fig(self, ds: Dataset, loss_te) -> go.Figure:
        """Mean test loss across quantile bins of each feature (univariate view)."""
        X = ds.X_test
        feats = ds.feature_names
        pick = feats[: min(6, len(feats))]
        fig = go.Figure()
        for f in pick:
            col = X[f].to_numpy(float)
            try:
                q = pd.qcut(col, q=min(6, len(np.unique(col))), duplicates="drop")
            except Exception:
                continue
            grp = pd.DataFrame({"loss": loss_te, "bin": q}).groupby("bin", observed=True)["loss"].mean()
            centers = [iv.mid for iv in grp.index]
            fig.add_trace(go.Scatter(x=centers, y=grp.values, mode="lines+markers", name=f))
        fig.add_hline(y=float(np.mean(loss_te)), line=dict(color=MUTED, dash="dot"),
                      annotation_text="global mean loss")
        fig.update_xaxes(title="feature value (quantile bin centers)")
        fig.update_yaxes(title="mean test loss")
        return _base(fig, "Univariate error profiles", height=440)

    def _landscape_fig(self, density_all, noise_all, loss_te, g_density) -> go.Figure:
        """AMIF-style two-axis view (cheap): x = sparsity (neighbour distance),
        y = label noise (irreducibility), colour = test loss. Bottom-left = dense
        + clean; top-right = sparse + noisy."""
        fig = go.Figure(go.Scatter(
            x=density_all, y=noise_all, mode="markers",
            marker=dict(size=5, color=loss_te, colorscale="OrRd", showscale=True,
                        colorbar=dict(title="loss")),
            hovertemplate="sparsity=%{x:.2f}<br>noise=%{y:.2f}"
                          "<br>loss=%{marker.color:.3f}<extra></extra>"))
        fig.add_vline(x=1.5 * g_density, line=dict(color=WARN, dash="dot"),
                      annotation_text="sparse")
        fig.add_hline(y=0.35, line=dict(color=MUTED, dash="dot"),
                      annotation_text="noisy")
        fig.update_xaxes(title="data sparsity  (mean kNN distance →)")
        fig.update_yaxes(title="label noise  (kDN disagreement →)")
        return _base(fig, "Weakness landscape (density × signal, AMIF-style)",
                         height=460)

    def _drivers_fig(self, drivers: pd.DataFrame) -> go.Figure:
        d = drivers.head(10).iloc[::-1]
        fig = go.Figure(go.Bar(x=d["importance"], y=d["feature"], orientation="h",
                               marker_color=ACCENT))
        fig.update_xaxes(title="error-tree importance")
        return _base(fig, "Error drivers (features that predict where loss is high)",
                         height=360)

    def _verdict(self, slices, g_score, task, thr) -> str:
        name = headline_name(task)
        weak = slices[slices["likely_cause"] != "(healthy)"]
        if weak.empty:
            return (f"No region beyond the adaptive threshold (loss ratio > {thr:.2f}) "
                    f"vs global {name}={g_score:.3f}. Error looks evenly spread — no "
                    "obvious local/regional overfitting.")
        counts = weak["likely_cause"].value_counts().to_dict()
        worst = weak.iloc[0]
        parts = [f"Global {name}={g_score:.3f}. Adaptive weak threshold = loss ratio "
                 f"> {thr:.2f}. Found {len(weak)} weak region(s). "
                 f"Worst: `{worst['rule']}` — {worst['n_test']} test rows, "
                 f"loss ×{worst['loss_ratio']} vs global, attributed to "
                 f"{worst['likely_cause']}."]
        by = ", ".join(f"{k}: {v}" for k, v in counts.items())
        parts.append("Cause breakdown — " + by + ".")
        if "Local overfitting" in counts:
            parts.append("Local overfitting present → regularize/prune where train "
                         "loss is low but test loss spikes.")
        if "Distribution shift" in counts:
            parts.append("Shift present → reweight or retrain on recent data.")
        if "Data sparsity" in counts:
            parts.append("Sparsity present → collect/upsample the flagged regions.")
        return " ".join(parts)


# --------------------------------------------------------------------------- #
# rule extraction / matching helpers                                          #
# --------------------------------------------------------------------------- #
def _leaf_rules(tree, feature_names) -> dict:
    t = tree.tree_
    rules: dict[int, list] = {}

    def recurse(node, conds):
        if t.children_left[node] == -1:
            rules[node] = dict(conds)  # copy
            return
        f = feature_names[t.feature[node]]
        thr = t.threshold[node]
        left = dict(conds)
        left[f] = (left.get(f, (-np.inf, np.inf))[0], min(left.get(f, (-np.inf, np.inf))[1], thr))
        recurse(t.children_left[node], left)
        right = dict(conds)
        right[f] = (max(right.get(f, (-np.inf, np.inf))[0], thr), right.get(f, (-np.inf, np.inf))[1])
        recurse(t.children_right[node], right)

    recurse(0, {})
    return {lid: _fmt_rule(bounds) for lid, bounds in rules.items()}


def _fmt_rule(bounds: dict) -> str:
    parts = []
    for f, (lo, hi) in bounds.items():
        if lo == -np.inf:
            parts.append(f"{f}≤{hi:.3g}")
        elif hi == np.inf:
            parts.append(f"{f}>{lo:.3g}")
        else:
            parts.append(f"{lo:.3g}<{f}≤{hi:.3g}")
    return " & ".join(parts) if parts else "all"


def _parse_rule(rule: str) -> list:
    """Parse formatted rule back into (feat, lo, hi) constraints for matching."""
    out = []
    if rule == "all":
        return out
    for clause in rule.split(" & "):
        if "<" in clause and "≤" in clause and clause.count("≤") == 1 and "<" in clause.split("≤")[0]:
            lo_s, rest = clause.split("<", 1)
            feat, hi_s = rest.split("≤")
            out.append((feat, float(lo_s), float(hi_s)))
        elif "≤" in clause:
            feat, hi_s = clause.split("≤")
            out.append((feat, -np.inf, float(hi_s)))
        elif ">" in clause:
            feat, lo_s = clause.split(">")
            out.append((feat, float(lo_s), np.inf))
    return out


def _row_matches(rule: str, row: dict) -> bool:
    for feat, lo, hi in _parse_rule(rule):
        v = row.get(feat, np.nan)
        if not (lo < v <= hi):
            return False
    return True


def _tint(hex_color: str) -> str:
    """Light background tint of a cause color."""
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    r, g, b = [int(c + (255 - c) * 0.72) for c in (r, g, b)]
    return f"rgb({r},{g},{b})"


In [41]:
"""End-to-end demo of model_eval_toolkit.

Builds:
  * a classification set with a PLANTED weak region (label noise in a corner),
  * a DELIBERATELY overfit model (deep tree) to light up complexity signals,
  * a time-ordered, DRIFTING fraud-like stream for the temporal curve.

Also shows the 'pydantic dataloader' entry point via a tiny loader class.
"""

"End-to-end demo of model_eval_toolkit.\n\nBuilds:\n  * a classification set with a PLANTED weak region (label noise in a corner),\n  * a DELIBERATELY overfit model (deep tree) to light up complexity signals,\n  * a time-ordered, DRIFTING fraud-like stream for the temporal curve.\n\nAlso shows the 'pydantic dataloader' entry point via a tiny loader class.\n"

In [42]:
import numpy as np
import pandas as pd
from pydantic import BaseModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier


In [43]:




rng = np.random.default_rng(0)


# --------------------------------------------------------------------------- #
# 1. A "pydantic dataloader" — any object that can yield a DataFrame works.    #
# --------------------------------------------------------------------------- #
class FrameLoader(BaseModel):
    """Stand-in for a real pydantic dataloader; exposes .load() -> DataFrame."""
    model_config = {"arbitrary_types_allowed": True}
    frame: pd.DataFrame

    def load(self) -> pd.DataFrame:
        return self.frame


def make_classification_with_weak_region(n):
    x0 = rng.normal(0, 1, n)
    x1 = rng.normal(0, 1, n)
    x2 = rng.normal(0, 1, n)
    logit = 1.5 * x0 - 1.0 * x1 + 0.5 * x0 * x1
    p = 1 / (1 + np.exp(-logit))
    y = (rng.uniform(size=n) < p).astype(int)
    # PLANT 1 — irreducible noise: a top strip (x1>1.0) gets ~random labels.
    noisy = x1 > 1.0
    y[noisy] = (rng.uniform(size=noisy.sum()) < 0.5).astype(int)
    return pd.DataFrame({"x0": x0, "x1": x1, "x2": x2, "y": y})


def drop_training_support(train_df, mask_col="x0", thresh=1.3, keep=0.08):
    """PLANT 2 — data sparsity: keep only a few TRAIN rows where x0>thresh."""
    hi = train_df[mask_col] > thresh
    keep_idx = train_df[hi].sample(frac=keep, random_state=1).index
    return pd.concat([train_df[~hi], train_df.loc[keep_idx]]).sort_index()


def make_drifting_fraud(n):
    """Fraud signal whose direction ROTATES over time -> old data is stale."""
    ts = np.sort(rng.uniform(0, 1, n))          # time in [0,1]
    a = rng.normal(0, 1, n)
    b = rng.normal(0, 1, n)
    theta = 2.5 * ts                            # signal rotates with time
    signal = np.cos(theta) * a + np.sin(theta) * b
    p = 1 / (1 + np.exp(-(2.2 * signal - 1.0)))  # base rate < 0.5
    y = (rng.uniform(size=n) < p).astype(int)
    return pd.DataFrame({"amount": a, "velocity": b, "ts": ts, "is_fraud": y})


# --------------------------------------------------------------------------- #
print("=" * 70)
print("PART A — classification, overfit model, planted weak region")
print("=" * 70)
df = make_classification_with_weak_region(9000)
train_df, test_df = df.iloc[:6000].copy(), df.iloc[6000:].copy()
train_df = drop_training_support(train_df)     # thin out x0>1.3 in TRAIN only
train_loader = FrameLoader(frame=train_df)   # <- pydantic dataloader
test_loader = FrameLoader(frame=test_df)

ds = Dataset.from_loaders(train_loader, test_loader, target="y",
                          task="classification")
cfg = EvalConfig(task=Task.CLASSIFICATION, n_repeats=10, n_bootstrap=15,
                 max_slice_depth=3, min_slice_frac=0.04)

overfit_model = DecisionTreeClassifier(max_depth=None, min_samples_leaf=1,
                                       random_state=0).fit(
    ds.X_train.to_numpy(float), ds.y_train.to_numpy())




PART A — classification, overfit model, planted weak region


In [44]:

# A regularized model: global error is controlled, so REGIONAL problems stand out.
good_model = RandomForestClassifier(n_estimators=200, min_samples_leaf=25,
                                     random_state=0).fit(
    ds.X_train.to_numpy(float), ds.y_train.to_numpy())

In [47]:
print("\n[Weak points — on the regularized model]")
wrep, drivers = WeakpointAnalyzer(cfg).analyze(good_model, ds)



[Weak points — on the regularized model]


In [48]:
drivers

,feature,importance
0,x0,0.603
1,x1,0.395
2,x2,0.002


In [ ]:
wrep.slices

,rule,n_test,test_share,train_share,local_score,global_score,local_test_loss,local_train_loss,loss_ratio,overfit_gap,coverage,local_pred_var,loss_std,uncertainty,likely_cause
0,1<x1≤1.66 & x2≤-0.384,120,0.040,0.040,0.423,0.781,0.734,0.647,1.31,0.087,1.01,0.0076,0.180,0.424,Elevated error
1,1<x1≤1.66 & x2>-0.384,218,0.073,0.072,0.516,0.781,0.701,0.654,1.25,0.046,0.99,0.0035,0.139,0.444,(healthy)
2,x1>1.66,130,0.043,0.048,0.617,0.781,0.678,0.649,1.21,0.029,1.12,0.0054,0.160,0.430,(healthy)
3,x1≤1 & -1.09<x0≤0.774,1625,0.542,0.598,0.734,0.781,0.606,0.548,1.08,0.058,1.10,0.0407,0.392,0.328,(healthy)
4,-0.225<x1≤1 & x0>0.774,309,0.103,0.061,0.494,0.781,0.488,0.493,0.87,-0.005,0.59,0.0050,0.429,0.263,(healthy)
5,x1≤1 & x0≤-1.09,319,0.106,0.123,0.813,0.781,0.390,0.376,0.70,0.013,1.16,0.0290,0.425,0.221,(healthy)
6,x1≤-0.225 & x0>0.774,279,0.093,0.057,0.579,0.781,0.313,0.353,0.56,-0.040,0.61,0.0011,0.461,0.159,(healthy)


0.9999999999999999

In [ ]:
"""
LearningCurveAnalyzer
=====================
Two things sklearn.learning_curve does NOT give you:

A) SATURATION / SUFFICIENCY (random mode)
   We fit an inverse-power law  err(n) = c + a * n^(-b)  to the *validation*
   error and extrapolate. That yields actionable numbers:
     - asymptotic error c (the floor this model+data can reach),
     - how much error is still on the table,
     - how many more samples to capture 90% of the remaining gain,
     - a regime label: high-bias / high-variance / saturated.
   So instead of "here's a curve", you get "you are data-limited, ~2x the data
   buys you ~30% of the remaining error" or "adding data won't help, raise
   capacity".

B) TEMPORAL / DRIFT (temporal mode)  -- the fraud-detection case
   Random subsampling silently assumes i.i.d. data. Under drift, *older* rows
   can be worse than useless. We fix the validation block to the MOST RECENT
   data and grow the training window BACKWARD in time. If the validation score
   peaks at a limited look-back and then decays as more history is added, older
   data is detrimental — we report the optimal look-back window W*.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.optimize import curve_fit
from sklearn.base import clone

from .data import Dataset, EvalConfig, Task
from .metrics import headline_name, score
from .modeva_ext import drift_drivers, resilience_stress_curve
from . import viz


def _err(headline: float, task: Task) -> float:
    """Turn a 'higher is better' score into an error in [0,1]."""
    return float(np.clip(1 - headline, 0, 1))


def _power_law(n, c, a, b):
    return c + a * np.power(n, -b)


# --------------------------------------------------------------------------- #
@dataclass
class LearningCurveReport:
    mode: str
    table: pd.DataFrame
    fig: go.Figure
    verdict: str
    details: dict = field(default_factory=dict)

    def show(self):
        self.fig.show()


class LearningCurveAnalyzer:
    def __init__(self, config: Optional[EvalConfig] = None):
        self.cfg = config or EvalConfig()
        self.rng = np.random.default_rng(self.cfg.random_state)

    # ===================================================================== #
    # A) RANDOM MODE with saturation extrapolation                          #
    # ===================================================================== #
    def random_curve(self, model, ds: Dataset,
                     fractions=(0.1, 0.2, 0.35, 0.5, 0.7, 0.85, 1.0),
                     repeats: int = 3) -> LearningCurveReport:
        Xtr, ytr = ds.X_train.to_numpy(float), ds.y_train.to_numpy()
        Xte, yte = ds.X_test.to_numpy(float), ds.y_test.to_numpy()
        n_full = len(ytr)
        rows = []
        for frac in fractions:
            n = max(20, int(frac * n_full))
            tr_s, va_s = [], []
            for _ in range(repeats):
                idx = self.rng.choice(n_full, size=min(n, n_full), replace=False)
                m = clone(model).fit(Xtr[idx], ytr[idx])
                tr_s.append(score(ytr[idx], m, Xtr[idx], ds.task))
                va_s.append(score(yte, m, Xte, ds.task))
            rows.append(dict(n=n, train=np.mean(tr_s), val=np.mean(va_s),
                             val_std=np.std(va_s)))
        curve = pd.DataFrame(rows)

        # Fit saturation law on VALIDATION error.
        ns = curve["n"].to_numpy(float)
        val_err = 1 - curve["val"].to_numpy(float)
        c_hat, extra = self._fit_saturation(ns, val_err, n_full)

        fig = self._random_fig(curve, ds.task, extra)
        table, verdict = self._random_table_verdict(curve, ds.task, c_hat, extra, n_full)
        return LearningCurveReport("random", table, fig, verdict,
                                   details=dict(curve=curve, **extra))

    def _fit_saturation(self, ns, val_err, n_full):
        try:
            p0 = [max(val_err.min(), 1e-3), max(val_err.max() - val_err.min(), 1e-3), 0.5]
            bounds = ([0, 0, 0.05], [1, 5, 3.0])
            popt, _ = curve_fit(_power_law, ns, val_err, p0=p0, bounds=bounds, maxfev=8000)
            c, a, b = popt
            cur_err = _power_law(n_full, *popt)
            floor = c
            remaining = max(cur_err - floor, 0)
            # n to capture 90% of the remaining gain from n_full
            target = floor + 0.1 * remaining
            n_needed = (a / max(target - c, 1e-6)) ** (1 / b) if target > c else np.inf
            details = dict(fit_ok=True, c=float(c), a=float(a), b=float(b),
                           cur_err=float(cur_err), floor=float(floor),
                           remaining=float(remaining),
                           n_for_90pct=float(min(n_needed, 1e12)))
            return float(c), details
        except Exception as e:  # pragma: no cover
            return float(val_err.min()), dict(fit_ok=False, reason=str(e))

    def _random_fig(self, curve, task, extra):
        name = headline_name(task)
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=curve["n"], y=curve["train"], name=f"train {name}",
                                 mode="lines+markers", line=dict(color=viz.MUTED)))
        fig.add_trace(go.Scatter(
            x=curve["n"], y=curve["val"], name=f"validation {name}",
            mode="lines+markers", line=dict(color=viz.ACCENT),
            error_y=dict(type="data", array=curve["val_std"], visible=True)))
        if extra.get("fit_ok"):
            xs = np.linspace(curve["n"].min(), curve["n"].max() * 3, 100)
            ys = 1 - _power_law(xs, extra["c"], extra["a"], extra["b"])
            fig.add_trace(go.Scatter(x=xs, y=ys, name="extrapolation",
                                     mode="lines", line=dict(color=viz.WARN, dash="dash")))
            fig.add_hline(y=1 - extra["floor"], line=dict(color=viz.GOOD, dash="dot"),
                          annotation_text="estimated ceiling")
        fig.update_xaxes(title="training samples (n)")
        fig.update_yaxes(title=name)
        return viz._base(fig, "Learning curve with saturation extrapolation", height=460)

    def _random_table_verdict(self, curve, task, c_hat, extra, n_full):
        name = headline_name(task)
        last = curve.iloc[-1]
        gap = last["train"] - last["val"]
        rows = [
            ("current n", f"{int(last['n'])}", ""),
            (f"train {name}", f"{last['train']:.3f}", ""),
            (f"val {name}", f"{last['val']:.3f}", ""),
            ("train-val gap", f"{gap:.3f}", "variance signal"),
        ]
        verdict_bits = []
        if extra.get("fit_ok"):
            ceil = 1 - extra["floor"]
            rows += [
                (f"estimated ceiling {name}", f"{ceil:.3f}", "best this model can do"),
                ("remaining error to floor", f"{extra['remaining']:.3f}", ""),
                ("n for 90% of remaining gain", f"{extra['n_for_90pct']:.0f}",
                 f"~{extra['n_for_90pct']/max(n_full,1):.1f}x current data"),
            ]
            if gap > 0.08 and extra["remaining"] > 0.01:
                verdict_bits.append(
                    "HIGH-VARIANCE / data-limited: a visible train-val gap and the "
                    "curve is still descending. More data should help — roughly "
                    f"{extra['n_for_90pct']/max(n_full,1):.1f}x buys most of the "
                    "remaining error.")
            elif extra["remaining"] <= 0.01:
                verdict_bits.append(
                    "SATURATED: validation error has essentially reached its floor. "
                    "More rows of the same kind won't help — raise model capacity or "
                    "add informative features instead.")
            else:
                verdict_bits.append(
                    "MILD returns: the curve is flattening; extra data yields small "
                    "gains. Weigh collection cost against the modest expected lift.")
            if last["train"] - last["val"] < 0.03 and last["val"] < 0.6 \
                    and task == Task.CLASSIFICATION:
                verdict_bits.append(
                    "Train and val are both low and close → HIGH-BIAS: the model "
                    "underfits; more data alone won't fix it.")
        else:
            verdict_bits.append("Saturation fit failed; read the curve directly.")
        return pd.DataFrame(rows, columns=["quantity", "value", "note"]), " ".join(verdict_bits)

    # ===================================================================== #
    # B) TEMPORAL MODE — grow the training window by CALENDAR MONTHS         #
    #    (months of history, extending backward from the most recent one)   #
    # ===================================================================== #
    def temporal_curve(self, model, ds: Dataset, *, freq: str = "M",
                       val_periods: int = 1, n_periods: Optional[int] = None,
                       min_period_samples: int = 30) -> LearningCurveReport:
        """Does older history help or hurt, measured in whole calendar periods?

        The most recent `val_periods` period(s) are held out as validation; the
        training window then grows BACKWARD one period at a time (last 1 month,
        last 2 months, ...). If validation peaks at a limited look-back and
        decays as older months are added, old data is stale.

        freq : pandas offset alias when the time column is datetime — "M" month
               (default), "W" week, "Q" quarter. If the time column is numeric
               (not dates), the range is split into `n_periods` equal-width bins
               labelled generically and treated as ordered periods.
        """
        if ds.time_train is None:
            raise ValueError("temporal_curve needs a time column (set config.time_col "
                             "and build the Dataset with time_col=...).")
        pid, labels, unit = self._period_index(
            ds.time_train, freq, n_periods or self.cfg.n_windows)
        P = len(labels)
        if P < 3:
            raise ValueError(
                f"Need at least 3 time periods to trace a temporal curve; found {P} "
                f"{unit}(s). Provide a datetime time_col spanning several {unit}s, "
                "increase n_periods for numeric time, or use random_curve().")
        if val_periods >= P - 1:
            val_periods = 1

        X = ds.X_train.to_numpy(float)
        y = ds.y_train.to_numpy()
        val_mask = pid >= (P - val_periods)          # most recent period(s)
        Xval, yval = X[val_mask], y[val_mask]
        pool_mask = ~val_mask
        last_tp = P - val_periods - 1                # newest training period id

        # Grow look-back L = 1..(#training periods): most recent L training periods.
        rows = []
        for L in range(1, last_tp + 2):
            lo = last_tp - L + 1
            m = pool_mask & (pid >= lo) & (pid <= last_tp)
            if m.sum() < min_period_samples:
                continue
            mdl = clone(model).fit(X[m], y[m])
            rows.append(dict(lookback=L, n_samples=int(m.sum()),
                             from_period=labels[lo], to_period=labels[last_tp],
                             val=score(yval, mdl, Xval, ds.task)))
        tc = pd.DataFrame(rows)
        if tc.empty:
            raise ValueError("No period had enough samples; lower min_period_samples "
                             "or widen the period (e.g. freq='Q').")

        # Contrast: newest single training period vs oldest single training period.
        newest_m = pool_mask & (pid == last_tp)
        oldest_m = pool_mask & (pid == 0)
        s_new = (score(yval, clone(model).fit(X[newest_m], y[newest_m]), Xval, ds.task)
                 if newest_m.sum() >= min_period_samples else np.nan)
        s_old = (score(yval, clone(model).fit(X[oldest_m], y[oldest_m]), Xval, ds.task)
                 if oldest_m.sum() >= min_period_samples else np.nan)

        best = int(tc["val"].idxmax())
        w_star = int(tc.loc[best, "lookback"])          # optimal look-back IN PERIODS
        s_star = float(tc.loc[best, "val"])
        s_full = float(tc.iloc[-1]["val"])
        max_lb = int(tc["lookback"].max())
        decay = s_star - s_full                          # >0 => oldest months hurt

        feats = ds.feature_names
        drivers = drift_drivers(pd.DataFrame(X[oldest_m], columns=feats),
                                pd.DataFrame(Xval, columns=feats), feats)

        fig = self._temporal_fig(tc, ds.task, w_star, max_lb, unit)
        table, verdict = self._temporal_table_verdict(
            ds.task, s_new, s_old, w_star, s_star, s_full, decay, max_lb, unit,
            labels[0], labels[last_tp], labels[P - val_periods], labels[-1])
        top = drivers[drivers["psi"] > 0.1]
        if not top.empty:
            names = ", ".join(f"{r.feature} (PSI={r.psi})" for r in top.head(3).itertuples())
            verdict += f" Feature drift (covariate) is led by: {names}."
        else:
            verdict += (" No strong covariate (feature-distribution) drift — any drift "
                        "above is concept drift in P(y|x), which the temporal refit "
                        "curve captures but PSI/KS cannot.")
        return LearningCurveReport("temporal", table, fig, verdict,
                                   details=dict(curve=tc, unit=unit, periods=labels,
                                                w_star=w_star, s_star=s_star,
                                                s_full=s_full, decay=decay,
                                                s_new=s_new, s_old=s_old,
                                                drift_drivers=drivers))

    @staticmethod
    def _period_index(time_series, freq: str, n_periods: int):
        """Map each row to an ordered period id. Calendar buckets when the column
        is genuine datetime; equal-width numeric bins otherwise.
        Returns (period_id_per_row, ordered_labels, unit_word)."""
        s = pd.Series(np.asarray(time_series))
        unit_word = {"M": "month", "W": "week", "Q": "quarter", "Y": "year"}.get(freq, "period")
        is_dt = pd.api.types.is_datetime64_any_dtype(s)
        if not is_dt and not pd.api.types.is_numeric_dtype(s):
            parsed = pd.to_datetime(s, errors="coerce")
            if parsed.notna().mean() > 0.99 and \
                    (parsed.max() - parsed.min()) > pd.Timedelta(days=28):
                s, is_dt = parsed, True
        if is_dt:
            per = pd.to_datetime(s).dt.to_period(freq)
            uniq = sorted(per.unique())
            mapping = {p: i for i, p in enumerate(uniq)}
            return per.map(mapping).to_numpy(), [str(p) for p in uniq], unit_word
        # numeric fallback: equal-width bins over the time range
        v = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
        edges = np.linspace(np.nanmin(v), np.nanmax(v), n_periods + 1)
        pid = np.clip(np.digitize(v, edges[1:-1]), 0, n_periods - 1)
        labels = [f"P{i + 1}" for i in range(n_periods)]
        return pid, labels, "period"

    # ===================================================================== #
    # C) STRESS MODE — resilience to worst-case drift (refit-free, no time)  #
    # ===================================================================== #
    def stress_curve(self, model, ds: Dataset, scenario: str = "worst-sample",
                     levels=(0.0, 0.1, 0.2, 0.3, 0.4, 0.5)) -> LearningCurveReport:
        """Refit-free resilience stress test (Modeva-inspired). Progressively
        oversamples a hard subpopulation of the test set and tracks the metric.
        `scenario` ∈ {worst-sample, worst-cluster, edge, hard-sample}."""
        curve, fig, verdict = resilience_stress_curve(
            model, ds, scenario=scenario, levels=levels,
            n_clusters=self.cfg.n_windows, random_state=self.cfg.random_state)
        return LearningCurveReport("stress", curve, fig, verdict,
                                   details=dict(curve=curve, scenario=scenario))

    def _temporal_fig(self, tc, task, w_star, max_lb, unit):
        name = headline_name(task)
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=tc["lookback"], y=tc["val"], mode="lines+markers",
            name=f"val {name} on most-recent {unit}",
            line=dict(color=viz.ACCENT),
            customdata=np.stack([tc["from_period"], tc["to_period"], tc["n_samples"]], axis=-1),
            hovertemplate=("look-back=%{x} " + unit + "s<br>%{customdata[0]}→%{customdata[1]}"
                           "<br>n=%{customdata[2]}<br>" + name + "=%{y:.3f}<extra></extra>")))
        fig.add_vline(x=w_star, line=dict(color=viz.GOOD, dash="dash"),
                      annotation_text=f"optimal look-back = {w_star} {unit}s")
        fig.add_annotation(x=max_lb, y=tc["val"].iloc[-1],
                           text="all history", showarrow=True, arrowhead=2)
        fig.update_xaxes(title=f"training window = most recent N {unit}s (grows backward)",
                         dtick=1)
        fig.update_yaxes(title=f"validation {name}")
        return viz._base(fig, f"Temporal learning curve — months of history "
                              f"vs performance", height=460)

    def _temporal_table_verdict(self, task, s_new, s_old, w_star, s_star, s_full,
                                decay, max_lb, unit, oldest_lbl, newest_train_lbl,
                                val_start_lbl, val_end_lbl):
        name = headline_name(task)
        rec = (f"{s_new - s_old:+.3f}" if (s_new == s_new and s_old == s_old) else "n/a")
        rows = [
            (f"validation window", f"{val_start_lbl}…{val_end_lbl}", "the 'present'"),
            (f"newest {unit} ({newest_train_lbl}) → val {name}",
             f"{s_new:.3f}" if s_new == s_new else "n/a", "recency value"),
            (f"oldest {unit} ({oldest_lbl}) → val {name}",
             f"{s_old:.3f}" if s_old == s_old else "n/a", "antiquity value"),
            ("recency advantage", rec, "new minus old"),
            (f"optimal look-back", f"{w_star} {unit}s", f"of {max_lb} available"),
            (f"val {name} at optimum", f"{s_star:.3f}", ""),
            (f"val {name} using ALL history", f"{s_full:.3f}", f"{max_lb} {unit}s"),
            ("cost of using all history", f"{decay:+.3f}", "optimum minus all"),
        ]
        if decay > 0.01 and w_star < max_lb:
            verdict = (f"DRIFT DETECTED. Validation peaks at a {w_star}-{unit} look-back "
                       f"and DEGRADES by {decay:.3f} {name} when all {max_lb} {unit}s are "
                       f"used. Older months are stale/detrimental — cap the training "
                       f"window near {w_star} {unit}s (or add recency weighting / drift "
                       f"features).")
            if s_new == s_new and s_old == s_old:
                verdict += f" Recency advantage (newest vs oldest {unit}) is {rec}."
        elif s_new == s_new and s_old == s_old and s_new - s_old > 0.03:
            verdict = (f"MILD DRIFT: the most recent {unit} is more predictive than the "
                       f"oldest, but adding history doesn't hurt the aggregate. Keep all "
                       "months; consider time-decay weighting.")
        else:
            verdict = (f"STABLE: more months keep helping (or are neutral) and old months "
                       f"are about as useful as new. Use all available history; the "
                       "process looks stationary over this span.")
        return pd.DataFrame(rows, columns=["quantity", "value", "note"]), verdict

In [ ]:

# A regularized model: global error is controlled, so REGIONAL problems stand out.
good_model = RandomForestClassifier(n_estimators=200, min_samples_leaf=25,
                                     random_state=0).fit(
    ds.X_train.to_numpy(float), ds.y_train.to_numpy())



print("\n" + "=" * 70)
print("PART B — drifting fraud stream, temporal learning curve")
print("=" * 70)
fdf = make_drifting_fraud(7000)
ftr, fte = fdf.iloc[:5000], fdf.iloc[5000:]
fds = Dataset.from_frames(ftr, fte, target="is_fraud", task="classification",
                          time_col="ts")
fcfg = EvalConfig(task=Task.CLASSIFICATION, time_col="ts", n_windows=9)
fmodel = RandomForestClassifier(n_estimators=120, min_samples_leaf=15,
                                random_state=0).fit(
    fds.X_train.to_numpy(float), fds.y_train.to_numpy())
trep = LearningCurveAnalyzer(fcfg).temporal_curve(fmodel, fds)
print(trep.table.to_string(index=False))
print("VERDICT:", trep.verdict)

print("\n" + "=" * 70)
print("PART C — one-call HTML report")
print("=" * 70)
rep = OverfittingReport(cfg).run(overfit_model, ds)
path = rep.to_html("/home/claude/overfitting_report.html")
print("Wrote", path)

# also a temporal report for the fraud model
frep = OverfittingReport(fcfg).run(fmodel, fds)
fpath = frep.to_html("/home/claude/fraud_report.html")
print("Wrote", fpath)
print("\nDONE.")